# EN3150 Assignment 03: RealWaste

Run the cells in order. Edit the settings below to select models, learning rates, epochs, augmentation, and regularization. DATA_DIR can point directly to the nine RealWaste class folders; local and Colab extraction locations are detected automatically when it is empty.

The default run trains both custom CNNs and fine-tunes MobileNetV2 and EfficientNet-B0. Start with custom_models=["Model B"] and sota_models=[] for a shorter experiment. Keep at least 20 custom-model epochs for the assignment. Save each experiment to a different OUTPUT_DIR when comparing settings. First-time pretrained-model use downloads ImageNet weights.


In [ ]:
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [26]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader,Dataset,Subset
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix,recall_score, f1_score
import zipfile
from sklearn.model_selection import train_test_split
import time
import copy
import torch.optim as optim

In [ ]:
DATA_DIR = ""
DOWNLOAD_DATASET = True
IMAGE_SIZE = 64
SPLIT_SEED = 42
AUGMENT_TRAINING = True
OUTPUT_DIR = "results/my_run"
RUN_OPTIMIZER_COMPARISON = False

TRAINING_SETTINGS = {
    "custom_models": ["Model A", "Model B"],
    "sota_models": ["MobileNetV2", "EfficientNet-B0"],
    "custom_epochs": 30,
    "head_epochs": 5,
    "fine_tune_epochs": 20,
    "batch_size": 32,
    "custom_lr": 0.001,
    "head_lr": 0.001,
    "fine_tune_lr": 0.00001,
    "weight_decay": 0.0001,
    "label_smoothing": 0.05,
    "lr_patience": 3,
    "early_stopping_patience": 7,
    "num_workers": 0,
    "seed": 42,
    "pretrained": True,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
import json
import random
from pathlib import Path
from sklearn.metrics import classification_report


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_image_transforms(training=False, augment=True, image_size=64):
    operations = [transforms.Resize((image_size, image_size))]
    if training and augment:
        operations.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        ])
    operations.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transforms.Compose(operations)


def resolve_realwaste_path(data_dir="", download=True):
    configured = os.environ.get("REALWASTE_DIR", data_dir)
    if configured:
        path = Path(configured).expanduser()
        if not path.is_dir():
            raise FileNotFoundError(f"RealWaste class directory does not exist: {path}")
        return path
    candidates = [
        Path("/content/dataset/realwaste-main/RealWaste"),
        Path.cwd() / "data/realwaste-main/RealWaste",
        Path.cwd().parent / "data/realwaste-main/RealWaste",
    ]
    for path in candidates:
        if path.is_dir():
            return path
    archives = [
        Path("/content/drive/MyDrive/realwaste.zip"),
        Path.cwd() / "data/realwaste.zip",
        Path.cwd().parent / "data/realwaste.zip",
    ]
    archive = next((path for path in archives if path.is_file()), None)
    destination = Path("/content/dataset") if Path("/content").is_dir() else Path.cwd() / "data"
    if archive is None and not download:
        raise FileNotFoundError("Extract RealWaste and set DATA_DIR or REALWASTE_DIR to the folder containing its nine class folders.")
    destination.mkdir(parents=True, exist_ok=True)
    if archive is None:
        from urllib.request import urlretrieve
        archive = destination / "realwaste.zip"
        temporary_archive = destination / "realwaste.zip.part"
        print("Downloading RealWaste from UCI (about 657 MB)...", flush=True)
        urlretrieve("https://archive.ics.uci.edu/static/public/908/realwaste.zip", temporary_archive)
        temporary_archive.replace(archive)
    with zipfile.ZipFile(archive) as source:
        for member in source.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Archive contains a path outside the extraction directory")
        source.extractall(destination)
    return destination / "realwaste-main/RealWaste"


def prepare_realwaste_datasets(data_dir, image_size=64, augment=True, split_seed=42):
    if not 1 <= image_size <= 64:
        raise ValueError("Image size must be between 1 and 64 pixels")
    evaluation_data = datasets.ImageFolder(data_dir, transform=make_image_transforms(image_size=image_size))
    training_data = datasets.ImageFolder(data_dir, transform=make_image_transforms(True, augment, image_size))
    if evaluation_data.classes != training_data.classes or evaluation_data.samples != training_data.samples:
        raise ValueError("Training and evaluation datasets do not have identical image ordering")
    if len(evaluation_data.classes) != 9 or len(evaluation_data) != 4752:
        raise ValueError("Expected the full RealWaste dataset: 4,752 images in nine classes")
    indices = list(range(len(evaluation_data)))
    train_idx, temporary_idx, _, temporary_targets = train_test_split(
        indices, evaluation_data.targets, test_size=0.30,
        random_state=split_seed, stratify=evaluation_data.targets,
    )
    val_idx, test_idx = train_test_split(
        temporary_idx, test_size=0.50, random_state=split_seed, stratify=temporary_targets,
    )
    split_indices = {"train": train_idx, "val": val_idx, "test": test_idx}
    if not (set(train_idx).isdisjoint(val_idx) and set(train_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)):
        raise ValueError("Data splits overlap")
    split_data = {
        "train": Subset(training_data, train_idx),
        "val": Subset(evaluation_data, val_idx),
        "test": Subset(evaluation_data, test_idx),
    }
    return evaluation_data, split_data, split_indices


def make_dataloaders(split_data, batch_size=32, seed=42, num_workers=0):
    return {
        phase: DataLoader(
            subset, batch_size=batch_size, shuffle=phase == "train",
            generator=torch.Generator().manual_seed(seed + index),
            worker_init_fn=seed_worker, num_workers=num_workers,
            pin_memory=device.type == "cuda",
        )
        for index, (phase, subset) in enumerate(split_data.items())
    }


def train_model(model, dataloaders, criterion, optimizer, num_of_epochs=30,
                scheduler=None, patience=None, freeze_batchnorm=False, frozen_features=False):
    if num_of_epochs < 1:
        raise ValueError("Training requires at least one epoch")
    if patience is not None and patience < 1:
        raise ValueError("Early-stopping patience must be positive or None")
    history = {key: [] for key in ["train_loss", "train_acc", "val_loss", "val_acc", "epoch_seconds", "learning_rate"]}
    best_weights = copy.deepcopy(model.state_dict())
    best_accuracy = -float("inf")
    epochs_without_improvement = 0
    for epoch in range(num_of_epochs):
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        started = time.perf_counter()
        learning_rate = optimizer.param_groups[0]["lr"]
        for phase in ["train", "val"]:
            model.train(phase == "train")
            if phase == "train" and frozen_features:
                model.features.eval()
            if phase == "train" and freeze_batchnorm:
                for module in model.modules():
                    if isinstance(module, nn.modules.batchnorm._BatchNorm):
                        module.eval()
            loss_sum = 0.0
            correct = 0
            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                if phase == "train":
                    optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    if not torch.isfinite(loss):
                        raise FloatingPointError(f"Non-finite loss in {phase} at epoch {epoch + 1}")
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                loss_sum += loss.item() * inputs.size(0)
                correct += (outputs.argmax(dim=1) == labels).sum().item()
            count = len(dataloaders[phase].dataset)
            history[f"{phase}_loss"].append(loss_sum / count)
            history[f"{phase}_acc"].append(correct / count)
        if history["val_acc"][-1] > best_accuracy:
            best_accuracy = history["val_acc"][-1]
            best_weights = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if scheduler is not None:
            scheduler.step(history["val_loss"][-1])
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        history["epoch_seconds"].append(time.perf_counter() - started)
        history["learning_rate"].append(learning_rate)
        print(
            f"Epoch {epoch + 1}/{num_of_epochs} | "
            f"train loss={history['train_loss'][-1]:.4f} acc={history['train_acc'][-1]:.4f} | "
            f"val loss={history['val_loss'][-1]:.4f} acc={history['val_acc'][-1]:.4f} | lr={learning_rate:g}",
            flush=True,
        )
        if patience is not None and epochs_without_improvement >= patience:
            break
    model.load_state_dict(best_weights)
    model.eval()
    return model, history, float(np.mean(history["epoch_seconds"]))


def build_sota_model(model_name, num_classes, pretrained=True):
    if model_name == "MobileNetV2":
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v2(weights=weights)
        model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    elif model_name == "EfficientNet-B0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown pretrained architecture: {model_name}")
    return model


def get_sota_models(num_classes, pretrained=True):
    return (build_sota_model("MobileNetV2", num_classes, pretrained),
            build_sota_model("EfficientNet-B0", num_classes, pretrained))


def train_transfer_model(model, split_data, settings):
    criterion = nn.CrossEntropyLoss(label_smoothing=settings["label_smoothing"])
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True
    before_head = state_fingerprint(model.features.state_dict())
    optimizer = optim.Adam(model.classifier.parameters(), lr=settings["head_lr"], weight_decay=settings["weight_decay"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=settings["lr_patience"], min_lr=1e-7)
    model, head_history, _ = train_model(
        model, make_dataloaders(split_data, settings["batch_size"], settings["seed"], settings["num_workers"]),
        criterion, optimizer, settings["head_epochs"], scheduler=scheduler,
        patience=settings["early_stopping_patience"], freeze_batchnorm=True, frozen_features=True,
    )
    after_head = state_fingerprint(model.features.state_dict())
    if before_head != after_head:
        raise RuntimeError("Frozen backbone changed during classifier-head training")
    head_weights = copy.deepcopy(model.state_dict())
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            for parameter in module.parameters():
                parameter.requires_grad = False
    optimizer = optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=settings["fine_tune_lr"], weight_decay=settings["weight_decay"],
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=settings["lr_patience"], min_lr=1e-7)
    model, fine_history, _ = train_model(
        model, make_dataloaders(split_data, settings["batch_size"], settings["seed"] + 1, settings["num_workers"]),
        criterion, optimizer, settings["fine_tune_epochs"], scheduler=scheduler,
        patience=settings["early_stopping_patience"], freeze_batchnorm=True,
    )
    fine_changed = state_fingerprint(model.features.state_dict()) != after_head
    selected_stage = "fine_tune"
    if max(head_history["val_acc"]) >= max(fine_history["val_acc"]):
        model.load_state_dict(head_weights)
        selected_stage = "head"
    history = {key: head_history[key] + fine_history[key] for key in head_history}
    details = {
        "head_epochs": len(head_history["val_acc"]),
        "fine_tune_epochs": len(fine_history["val_acc"]),
        "head_best_val_accuracy": max(head_history["val_acc"]),
        "fine_tune_best_val_accuracy": max(fine_history["val_acc"]),
        "selected_stage": selected_stage,
        "head_backbone_unchanged": before_head == after_head,
        "fine_tune_backbone_changed": fine_changed,
    }
    return model.eval(), history, float(np.mean(history["epoch_seconds"])), details


def estimate_macs(model, image_size=64):
    counts = []
    handles = []

    def count(module, inputs, output):
        if isinstance(module, nn.Conv2d):
            counts.append(output.numel() * (module.in_channels // module.groups) * module.kernel_size[0] * module.kernel_size[1])
        elif isinstance(module, nn.Linear):
            counts.append(output.numel() * module.in_features)

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            handles.append(module.register_forward_hook(count))
    try:
        with torch.no_grad():
            model.eval()(torch.zeros(1, 3, image_size, image_size, device=device))
    finally:
        for handle in handles:
            handle.remove()
    return sum(counts)


def export_training_result(model, name, history, mean_seconds, test_loader, class_names, output_dir, details, image_size=64):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = output_dir / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    slug = name.lower().replace(" ", "_").replace("-", "_")
    labels, predictions = [], []
    with torch.no_grad():
        model.eval()
        for inputs, targets in test_loader:
            predictions.extend(model(inputs.to(device)).argmax(dim=1).cpu().tolist())
            labels.extend(targets.tolist())
    report = classification_report(labels, predictions, labels=list(range(len(class_names))), target_names=class_names, output_dict=True, zero_division=0)
    matrix = confusion_matrix(labels, predictions, labels=list(range(len(class_names))))
    checkpoint = checkpoint_dir / f"{slug}.pt"
    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, checkpoint)
    metrics = {
        "model": name,
        "total_parameters": sum(parameter.numel() for parameter in model.parameters()),
        "trainable_parameters_final_stage": count_parameters(model),
        "tensor_storage_kib": get_model_size_mb(model) * 1024,
        "checkpoint_file_kib": checkpoint.stat().st_size / 1024,
        "estimated_conv_linear_macs": estimate_macs(model, image_size),
        "epochs_completed": len(history["val_acc"]),
        "mean_epoch_seconds": mean_seconds,
        "test_accuracy": accuracy_score(labels, predictions),
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "history": history,
        "confusion_matrix": matrix.tolist(),
        "training_details": details,
    }
    (output_dir / f"{slug}_metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8")
    pd.DataFrame(report).transpose().to_csv(output_dir / f"{slug}_classification.csv")
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(output_dir / f"{slug}_confusion.csv")
    epochs = range(1, len(history["val_acc"]) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(epochs, history["train_loss"], label="Training")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Loss", title=f"{name}: training objective")
    axes[1].plot(epochs, np.array(history["train_acc"]) * 100, label="Training")
    axes[1].plot(epochs, np.array(history["val_acc"]) * 100, label="Validation")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title=f"{name}: accuracy")
    for axis in axes:
        if "head_epochs" in details:
            axis.axvline(details["head_epochs"] + 0.5, color="gray", linestyle="--", label="Fine-tuning begins")
        axis.legend()
        axis.grid(alpha=0.2)
    figure.tight_layout()
    figure.savefig(output_dir / f"{slug}_curves.png", dpi=150)
    plt.close(figure)
    figure, axis = plt.subplots(figsize=(10, 9))
    image = axis.imshow(matrix, cmap="Blues")
    axis.set_xticks(range(len(class_names)), labels=class_names, rotation=45, ha="right")
    axis.set_yticks(range(len(class_names)), labels=class_names)
    axis.set(xlabel="Predicted class", ylabel="Actual class", title=f"{name}: test confusion matrix")
    for row in range(len(class_names)):
        for column in range(len(class_names)):
            axis.text(column, row, str(matrix[row, column]), ha="center", va="center",
                      color="white" if matrix[row, column] > matrix.max() / 2 else "black")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    figure.tight_layout()
    figure.savefig(output_dir / f"{slug}_confusion.png", dpi=150)
    plt.close(figure)
    return metrics


def run_training_experiment(split_data, class_names, settings, output_dir, split_indices=None, image_size=64, augment=True, dataset_name="RealWaste"):
    if settings["custom_epochs"] < 1 or settings["head_epochs"] < 1 or settings["fine_tune_epochs"] < 1:
        raise ValueError("Epoch counts must be positive")
    if not 0 <= settings["label_smoothing"] < 1:
        raise ValueError("Label smoothing must be in [0, 1)")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    record = {
        "dataset": dataset_name, "classes": list(class_names), "settings": settings,
        "image_size": image_size, "augmentation_training_only": augment,
        "device": str(device), "torch_version": str(torch.__version__),
        "split_sizes": {key: len(value) for key, value in split_data.items()},
        "split_indices": split_indices, "models": {},
    }
    constructors = {"Model A": Standard_Model, "Model B": LightWeightModel}
    names = settings["custom_models"] + settings["sota_models"]
    if not names:
        raise ValueError("Select at least one model")
    if "Model A" in names and image_size != 64:
        raise ValueError("Model A's current classifier requires IMAGE_SIZE=64")
    for name in names:
        print(f"\nTraining {name} on {device}", flush=True)
        seed_experiment(settings["seed"])
        loaders = make_dataloaders(split_data, settings["batch_size"], settings["seed"], settings["num_workers"])
        if name in constructors:
            model = constructors[name](len(class_names)).to(device)
            if name == "Model B" and count_parameters(model) > 100_000:
                raise ValueError("Model B exceeds 100,000 trainable parameters")
            seed_experiment(settings["seed"])
            optimizer = optim.Adam(model.parameters(), lr=settings["custom_lr"], weight_decay=settings["weight_decay"])
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=settings["lr_patience"], min_lr=1e-7)
            model, history, mean_seconds = train_model(
                model, loaders, nn.CrossEntropyLoss(label_smoothing=settings["label_smoothing"]),
                optimizer, settings["custom_epochs"], scheduler=scheduler,
            )
            details = {"pretrained": False, "best_val_accuracy": max(history["val_acc"]), "selected_epoch": int(np.argmax(history["val_acc"])) + 1}
        else:
            model = build_sota_model(name, len(class_names), settings["pretrained"]).to(device)
            seed_experiment(settings["seed"])
            model, history, mean_seconds, details = train_transfer_model(model, split_data, settings)
            details["pretrained"] = settings["pretrained"]
        metrics = export_training_result(
            model, name, history, mean_seconds, loaders["test"], class_names, output_dir, details, image_size,
        )
        record["models"][name] = metrics
        (output_dir / "experiment.json").write_text(json.dumps(record, indent=2, allow_nan=False), encoding="utf-8")
        comparison = pd.DataFrame([{key: value for key, value in item.items() if key not in ["history", "confusion_matrix", "training_details"]} for item in record["models"].values()])
        comparison.to_csv(output_dir / "model_comparison.csv", index=False)
        print(f"{name}: test accuracy={metrics['test_accuracy']:.4f}, total parameters={metrics['total_parameters']:,}", flush=True)
    print(comparison.to_string(index=False))
    return record


In [ ]:
Data_Dir = resolve_realwaste_path(DATA_DIR, DOWNLOAD_DATASET)
print(f"RealWaste directory: {Data_Dir}")


In [ ]:
data_transforms = make_image_transforms(image_size=IMAGE_SIZE)
print(data_transforms)


In [ ]:
dataset, split_datasets, split_indices = prepare_realwaste_datasets(
    Data_Dir, image_size=IMAGE_SIZE, augment=AUGMENT_TRAINING, split_seed=SPLIT_SEED,
)
num_of_classes = len(dataset.classes)
print(f'Classes: {dataset.classes}')
print(f'Images: {len(dataset)}')
print({phase: len(subset) for phase, subset in split_datasets.items()})


In [ ]:
train_idx = split_indices['train']
val_idx = split_indices['val']
test_idx = split_indices['test']
train_dataset = split_datasets['train']
val_dataset = split_datasets['val']
test_dataset = split_datasets['test']
BATCH_SIZE = TRAINING_SETTINGS['batch_size']
dataloaders = make_dataloaders(split_datasets, BATCH_SIZE, TRAINING_SETTINGS['seed'], TRAINING_SETTINGS['num_workers'])


In [62]:
class Standard_Model(nn.Module):
  def __init__(self,num_of_classes):
    super(Standard_Model,self).__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size = 3, padding = 1),
        nn.ReLU(inplace = True),
        nn.MaxPool2d(kernel_size = 2, stride = 2),             # Here we have 32*32 and channels 16
        nn.Conv2d(16, 32,kernel_size = 3, padding = 1),        # Here channels become 32
        nn.ReLU(inplace = True),
        nn.MaxPool2d(kernel_size = 2, stride = 2),             # Here We have 16*16 and channels 32
        nn.Conv2d(32, 64, kernel_size = 3, padding = 1),       # Here Channels become 64
        nn.ReLU(inplace = True),
        nn.MaxPool2d(kernel_size = 2, stride = 2)             # Here We have 8*8 and channels 64

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),                   #Here we have 8*8*64
        nn.Linear(64*8*8, 128),
        nn.ReLU(inplace = True),
        nn.Linear(128, num_of_classes)   # Classify into one class within 9 classes
    )

  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x



In [50]:
class DepthWiseSeperableConvolution(nn.Module):
  def __init__(self, in_channels, out_channels, stride = 1):
    super(DepthWiseSeperableConvolution, self).__init__()
    self.depthwise = nn.Conv2d(
        in_channels,
        in_channels,
        kernel_size = 3,
        stride = stride,
        padding = 1,
        groups = in_channels,
        bias = False
    )
    self.pointwise = nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size = 1,
        bias = False
    )
    self.bn = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace = True)

  def forward(self,x):
    x = self.depthwise(x)
    x = self.pointwise(x)
    x = self.bn(x)
    return self.relu(x)


In [63]:
class LightWeightModel(nn.Module):
  def __init__(self, num_classes):
        super(LightWeightModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, stride=2, bias=False), # 32x32
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            DepthWiseSeperableConvolution(16, 32, stride=2), # 16x16
            DepthWiseSeperableConvolution(32, 64, stride=2), # 8x8
            DepthWiseSeperableConvolution(64, 64, stride=1), # 8x8
            nn.AdaptiveAvgPool2d((1, 1)) # 1x1
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )

  def forward(self, x):
      x = self.features(x)
      x = self.classifier(x)
      return x



In [37]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [38]:
def get_model_size_mb(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    size_all_mb = (param_size + buffer_size) / 1024**2
    return size_all_mb

### Training and checkpoint selection

The shared trainer records one complete training-plus-validation duration per epoch. It restores the first checkpoint with the highest validation accuracy. Custom models run the configured full epoch budget; transfer-learning stages can stop early using validation accuracy. ReduceLROnPlateau reduces the learning rate when validation loss stalls.

Random augmentation is restricted to the training subset. Validation and testing use deterministic resize and normalization. Label smoothing and Adam weight decay are adjustable in TRAINING_SETTINGS. These settings are new experiments; the historical baseline metrics below were recorded without these additions.


In [54]:
def evaluate_model(model, dataloader):
  model.eval()
  all_predictions = []
  all_labels = []

  with torch.no_grad():
    for inputs, labels in dataloader:
      inputs, labels = inputs.to(device), labels.to(device)
      outputs = model(inputs)
      temp, preds = torch.max(outputs,1)
      all_predictions.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  accuracy = accuracy_score(all_labels, all_predictions)
  precission = precision_score(all_labels, all_predictions, average='weighted',zero_division=0)
  recall = recall_score(all_labels, all_predictions, average='weighted',zero_division=0)
  f1 = f1_score(all_labels, all_predictions, average='weighted')
  confusion = confusion_matrix(all_labels, all_predictions)

  return accuracy, precission, recall, f1, confusion


In [29]:
def plot_history(history, title="Training and Validation Loss"):
    plt.figure(figsize=(8,6))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

### Two-stage lightweight transfer learning

MobileNetV2 and EfficientNet-B0 use ImageNet weights and nine-class classifier heads. Stage one trains only the classifier while keeping the feature extractor in evaluation mode. Stage two unfreezes backbone weights at a lower learning rate while keeping batch-normalization parameters and running statistics frozen. The better validation checkpoint across both stages is retained. All models use the same RealWaste split and 64 by 64 inputs.

References: [PyTorch transfer learning](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html), [MobileNetV2](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v2.html).


In [ ]:
import hashlib
import json
import random
from pathlib import Path
from sklearn.metrics import classification_report


def seed_experiment(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def state_fingerprint(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        digest.update(name.encode("utf-8"))
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


class RecordedRandomSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, seed):
        self.dataset = dataset
        self.generator = torch.Generator().manual_seed(seed)
        self.epoch_order_hashes = []

    def __iter__(self):
        indices = torch.randperm(len(self.dataset), generator=self.generator)
        digest = hashlib.sha256(indices.numpy().tobytes()).hexdigest()
        self.epoch_order_hashes.append(digest)
        return iter(indices.tolist())

    def __len__(self):
        return len(self.dataset)


def compare_optimizers(model_factory, train_data, val_data, epochs=30, seed=42, batch_size=32):
    configurations = [
        {"name": "Adam", "lr": 0.001, "momentum": None},
        {"name": "Adam", "lr": 0.0003, "momentum": None},
        {"name": "SGD", "lr": 0.01, "momentum": 0.0},
        {"name": "SGD", "lr": 0.1, "momentum": 0.0},
        {"name": "SGD + momentum", "lr": 0.01, "momentum": 0.9},
        {"name": "SGD + momentum", "lr": 0.1, "momentum": 0.9},
    ]
    seed_experiment(seed)
    initial_state = copy.deepcopy(model_factory().cpu().state_dict())
    initial_hash = state_fingerprint(initial_state)
    trials = []
    selected_model = None
    selected_key = None
    selected_index = None

    for index, configuration in enumerate(configurations):
        seed_experiment(seed)
        model = model_factory().to(device)
        model.load_state_dict(initial_state)
        assert state_fingerprint(model.state_dict()) == initial_hash
        sampler = RecordedRandomSampler(train_data, seed)
        loaders = {
            "train": DataLoader(
                train_data, batch_size=batch_size, sampler=sampler,
                generator=torch.Generator().manual_seed(seed + 1),
            ),
            "val": DataLoader(
                val_data, batch_size=batch_size, shuffle=False,
                generator=torch.Generator().manual_seed(seed + 2),
            ),
        }
        if configuration["name"] == "Adam":
            optimizer = optim.Adam(model.parameters(), lr=configuration["lr"])
        else:
            optimizer = optim.SGD(
                model.parameters(), lr=configuration["lr"],
                momentum=configuration["momentum"],
            )
        print(f"\nTrial {index + 1}/{len(configurations)}: {configuration}", flush=True)
        model, history, epoch_seconds = train_model(
            model, loaders, nn.CrossEntropyLoss(), optimizer, num_of_epochs=epochs,
        )
        assert all(len(values) == epochs for values in history.values())
        assert all(np.isfinite(values).all() for values in history.values())
        assert len(sampler.epoch_order_hashes) == epochs
        if trials:
            assert sampler.epoch_order_hashes == trials[0]["epoch_order_hashes"]
        best_index = int(np.argmax(history["val_acc"]))
        trial = {
            **configuration,
            "best_epoch": best_index + 1,
            "best_val_accuracy": history["val_acc"][best_index],
            "val_loss_at_best_epoch": history["val_loss"][best_index],
            "average_epoch_seconds": epoch_seconds,
            "initial_state_sha256": initial_hash,
            "epoch_order_hashes": sampler.epoch_order_hashes,
            "history": history,
        }
        trials.append(trial)
        key = (trial["best_val_accuracy"], -trial["val_loss_at_best_epoch"])
        if selected_key is None or key > selected_key:
            selected_key = key
            selected_index = index
            selected_model = copy.deepcopy(model).eval()
        print(
            f"Best validation accuracy: {trial['best_val_accuracy']:.4f} "
            f"at epoch {trial['best_epoch']}; mean epoch: {epoch_seconds:.3f}s",
            flush=True,
        )

    result = {
        "seed": seed,
        "epochs_per_trial": epochs,
        "batch_size": batch_size,
        "device": str(device),
        "torch_version": str(torch.__version__),
        "train_images": len(train_data),
        "validation_images": len(val_data),
        "selection_rule": "highest validation accuracy; lower validation loss breaks ties",
        "checkpoint_rule": "first epoch attaining each trial's highest validation accuracy",
        "selected_trial_index": selected_index,
        "trials": trials,
    }
    return result, selected_model


def optimizer_summary(result):
    rows = []
    for trial in result["trials"]:
        rows.append({
            "Optimizer": trial["name"],
            "Learning rate": trial["lr"],
            "Momentum": trial["momentum"],
            "Best validation accuracy (%)": 100 * trial["best_val_accuracy"],
            "Best epoch": trial["best_epoch"],
            "Validation loss at best epoch": trial["val_loss_at_best_epoch"],
            "Mean epoch seconds": trial["average_epoch_seconds"],
        })
    return pd.DataFrame(rows)


def save_optimizer_comparison(result, selected_model, test_data, class_names, output_dir, checkpoint_dir):
    output_dir = Path(output_dir)
    checkpoint_dir = Path(checkpoint_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    loader = DataLoader(test_data, batch_size=result["batch_size"], shuffle=False)
    labels = []
    predictions = []
    selected_model.eval()
    with torch.no_grad():
        for inputs, targets in loader:
            outputs = selected_model(inputs.to(device))
            predictions.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(targets.tolist())
    report = classification_report(
        labels, predictions, labels=list(range(len(class_names))),
        target_names=class_names, output_dict=True, zero_division=0,
    )
    confusion = confusion_matrix(labels, predictions, labels=list(range(len(class_names))))
    assert confusion.shape == (len(class_names), len(class_names))
    assert int(confusion.sum()) == len(test_data)
    checkpoint = checkpoint_dir / "selected_model_b.pt"
    torch.save({name: tensor.detach().cpu() for name, tensor in selected_model.state_dict().items()}, checkpoint)
    result["classes"] = list(class_names)
    result["test_images"] = len(test_data)
    result["selected_model"] = {
        "trainable_parameters": count_parameters(selected_model),
        "tensor_storage_kib": get_model_size_mb(selected_model) * 1024,
        "checkpoint_file_kib": checkpoint.stat().st_size / 1024,
        "test_accuracy": accuracy_score(labels, predictions),
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "confusion_matrix": confusion.tolist(),
    }
    assert result["selected_model"]["trainable_parameters"] <= 100_000
    (output_dir / "optimizer_results.json").write_text(
        json.dumps(result, indent=2, allow_nan=False), encoding="utf-8",
    )
    optimizer_summary(result).to_csv(output_dir / "optimizer_comparison.csv", index=False)
    pd.DataFrame(report).transpose().to_csv(output_dir / "selected_model_classification.csv")

    family_trials = []
    for family in ["Adam", "SGD", "SGD + momentum"]:
        candidates = [trial for trial in result["trials"] if trial["name"] == family]
        family_trials.append(max(candidates, key=lambda trial: (
            trial["best_val_accuracy"], -trial["val_loss_at_best_epoch"],
        )))
    epochs = range(1, result["epochs_per_trial"] + 1)
    figure_paths = []
    figure, axis = plt.subplots(figsize=(9, 5))
    for trial in family_trials:
        axis.plot(epochs, np.array(trial["history"]["val_acc"]) * 100,
                  label=f"{trial['name']} (lr={trial['lr']:g})")
    axis.set(xlabel="Epoch", ylabel="Validation accuracy (%)", title="Model B: optimizer convergence")
    axis.legend()
    axis.grid(alpha=0.2)
    figure.tight_layout()
    path = output_dir / "validation_accuracy.png"
    figure.savefig(path, dpi=160)
    figure_paths.append(path)
    plt.close(figure)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    for axis, trial in zip(axes, family_trials):
        axis.plot(epochs, trial["history"]["train_loss"], label="Training")
        axis.plot(epochs, trial["history"]["val_loss"], label="Validation")
        axis.set(title=f"{trial['name']}\nlr={trial['lr']:g}", xlabel="Epoch")
        axis.grid(alpha=0.2)
        axis.legend()
    axes[0].set_ylabel("Cross-entropy loss")
    figure.tight_layout()
    path = output_dir / "loss_curves.png"
    figure.savefig(path, dpi=160)
    figure_paths.append(path)
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(10, 9))
    image = axis.imshow(confusion, cmap="Blues")
    axis.set_xticks(range(len(class_names)), labels=class_names, rotation=45, ha="right")
    axis.set_yticks(range(len(class_names)), labels=class_names)
    axis.set(xlabel="Predicted class", ylabel="Actual class", title="Selected Model B: test confusion matrix")
    for row in range(len(class_names)):
        for column in range(len(class_names)):
            axis.text(column, row, str(confusion[row, column]), ha="center", va="center",
                      color="white" if confusion[row, column] > confusion.max() / 2 else "black")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    figure.tight_layout()
    path = output_dir / "confusion_matrix.png"
    figure.savefig(path, dpi=160)
    figure_paths.append(path)
    plt.close(figure)
    return figure_paths


In [ ]:
training_results = run_training_experiment(
    split_datasets, dataset.classes, TRAINING_SETTINGS, Path(OUTPUT_DIR),
    split_indices=split_indices, image_size=IMAGE_SIZE, augment=AUGMENT_TRAINING,
)

display(pd.read_csv(Path(OUTPUT_DIR) / "model_comparison.csv"))
for model_name in training_results["models"]:
    slug = model_name.lower().replace(" ", "_").replace("-", "_")
    display(Image.open(Path(OUTPUT_DIR) / f"{slug}_curves.png"))
    display(Image.open(Path(OUTPUT_DIR) / f"{slug}_confusion.png"))


## Recorded baseline experiments

The prior controlled 30-epoch Model B experiment selected Adam at learning rate 0.001: best validation accuracy 69.28% and test accuracy 71.11%. It used no augmentation, weight decay, label smoothing, or learning-rate schedule. Those results are historical measurements, not results for the new defaults above.

The complete histories, plots, metrics, and exact source snapshot are in results/optimizer_comparison/. The earlier 20-epoch custom-model runs are in results/custom_models/. Do not combine different experiments as if their settings were identical.


## Optimizer selection and tuning

Compare Model B using Adam (learning rates 0.001 and 0.0003), SGD (0.01 and 0.1), and SGD with momentum 0.9 (0.01 and 0.1). Every trial has 30 epochs, identical initial weights, identical training batches in the same order, the same 70/15/15 split, and no augmentation. Hashes verify the initial state and each epoch's sample order.

Choose each trial's checkpoint using validation accuracy. Select the optimizer and learning rate using highest validation accuracy, with lower validation loss breaking ties. Evaluate only that selected checkpoint on the test set. Optimizer comparisons therefore use validation metrics rather than repeated test-set tuning.

Adam adapts updates to individual parameters. Its smaller tested learning rates are compared with two SGD rates. The two SGD families use the same learning-rate grid so the effect of momentum can be examined at a fixed learning rate. This limited grid uses one random seed and does not establish a universal best optimizer.

Ordinary SGD follows the current batch's gradient. Momentum retains a decaying accumulation of previous gradients. A coefficient of 0.9 retains 90% of the previous momentum buffer before adding the current gradient. This can accelerate consistent directions and can also cause overshoot. Momentum needs one extra buffer per trainable parameter.

References: [PyTorch SGD](https://docs.pytorch.org/docs/2.14/generated/torch.optim.SGD.html), [PyTorch reproducibility](https://docs.pytorch.org/docs/2.14/notes/randomness.html). Exact numerical results can vary across software versions and devices.


In [ ]:
if RUN_OPTIMIZER_COMPARISON:
    optimizer_results, selected_model_b = compare_optimizers(
        lambda: LightWeightModel(num_of_classes), Subset(dataset, train_idx), val_dataset,
        epochs=30, seed=42, batch_size=32,
    )
    optimizer_figures = save_optimizer_comparison(
        optimizer_results, selected_model_b, test_dataset, dataset.classes,
        Path(OUTPUT_DIR) / "optimizer_comparison",
        Path(".local") / "optimizer_checkpoints",
    )
    print(optimizer_summary(optimizer_results).to_string(index=False))
    for figure_path in optimizer_figures:
        display(Image.open(figure_path))
else:
    print("The optional six-trial baseline comparison is disabled. Set RUN_OPTIMIZER_COMPARISON=True to rerun it.")
